# FullDrift final 100 clients, seed 42
Every train80% row; cosine T50. Session cap 2 epochs. All round weights retained; official test20% evaluated in separate notebooks. Headline round 50; no test-based selection.

In [1]:
import time
SESSION_T0=time.monotonic()

import os, subprocess, sys
# 0.28.0: 0.29 needs opentelemetry-api>=1.43 and upgrades the image's 1.38 (pip conflict lines, google-adk/otel-sdk pinned)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--progress-bar','off','wandb==0.28.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=_INLINE_WANDB_KEY or UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
os.environ['WANDB_API_KEY']=_wandb_key   # env login: no key-in-code / anonymous / netrc warnings
del _wandb_key
assert wandb.login(verify=True)
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: minhtriet (21522798-uit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B authenticated online


In [2]:
MH=['__init__.py', 'archs.py', 'channel.py', 'dagsnet.py', 'metrics.py', 'mobility.py', 'model_packet.py', 'new_fl.py', 'new_methods.py', 'p2.py', 'pilot.py', 'prepare.py', 'teacher_base.py', 'teacher_driver.py', 'v10_energy.py', 'v10_run.py', 'v2_math.py', 'v2_methods.py', 'v2_run.py', 'v3_dense.py', 'v3_math.py', 'v3_methods.py', 'v3_run.py', 'v4_mala.py', 'v4_run.py', 'v4_schema.py', 'v4_support.py', 'v5_anchor.py', 'v5_books.py', 'v5_run.py', 'v6_geometry.py', 'v6_run.py', 'v6_schema.py', 'v7_basis.py', 'v7_nystrom.py', 'v7_run.py', 'v8_hybrid.py', 'v8_run.py', 'v9_anchor.py', 'v9_run.py']
INPUTS={'__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'fd.py': '71fb2e5709fe0e3b52eae3b93a5c8ba6743748d1dab51241c2db34740481dacb', 'fd_eval.py': '1b125609c57a680d39209deca488c94ffa03beca2fa18be55cf06937b09b3ed0', 'fd_prepare.py': 'f60307b39a58a2045e66db7f704117758dc5014a1f87a9576356cc8750ec7869', 'full_driver.py': '66622498ae30f657a2e4b6baedfb645f12003809f63935f49d5551e564060f93', 'full_method.py': '2ac1b564f1ec1e03339f6c5c771727207125f8ed5f132d3d9e05237a20059ff2', 'full_run.py': '259de296cc080200ec8cb4948f19bd85043c409284bf3e3c711f0b607b25af86', 'meta.json': '1aeb0c9156c1f7d77694ffbee6f24a3f6abeec0c0dad48c1a45741cdb85fbff4', 'metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'scaler.json': '1b07ea714eee74d18f1304d3f18c50b1d025a6accfca5aa9b036c18164c02a37', 'teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', 'v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614'}
FINAL=True
CLIENTS=100
PARENT=None

import hashlib,json,shutil
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['TORCHINDUCTOR_COMPILE_THREADS']='1'
import torch
assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(2))
assert torch.__version__=='2.10.0+cu128'
hits=[p.parent for p in Path('/kaggle/input').rglob('input_manifest.json') if json.loads(p.read_text())==INPUTS]
assert len(hits)==1,('missing/ambiguous inputs',hits)
SRC=hits[0];WORK=Path('/kaggle/temp/fd');SOURCE=Path('/kaggle/working/source');OUT=Path('/kaggle/working/runs')
PROJ=WORK/'papers/full-drift/proj'
for folder in (WORK/'mh',WORK/'knowledge',PROJ,SOURCE,OUT):folder.mkdir(parents=True,exist_ok=True)
for name,digest in INPUTS.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==digest,name
    target=WORK/'mh'/name if name in MH else PROJ/name if name.endswith('.py') else WORK/'knowledge'/name
    shutil.copyfile(SRC/name,target);shutil.copyfile(SRC/name,SOURCE/name)
shutil.copyfile(SRC/'input_manifest.json',SOURCE/'input_manifest.json')
sys.path.insert(0,str(WORK));sys.path.insert(0,str(PROJ))
PARENT_DIR=None
if PARENT:   # validate/import before expensive fixture decode
    PARENT_DIR=Path('/kaggle/temp/parent')/PARENT['run']
    for rel,digest in PARENT['files'].items():
        flat=('resume_' if rel.startswith('resume/') else '')+Path(rel).name
        hits=[p for p in Path('/kaggle/input').rglob(flat) if hashlib.sha256(p.read_bytes()).hexdigest()==digest]
        assert len(hits)==1,('parent file',rel)
        target=PARENT_DIR/rel;target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(hits[0],target)
    print('parent',PARENT['run'],'round',PARENT['round'],flush=True)
t=time.monotonic()
from fd_prepare import prepare_fd
data=prepare_fd(WORK,final=FINAL,clients=CLIENTS)   # FINAL: every train80% row; separate evaluation covers every round
(SOURCE/'data_manifest.json').write_text(json.dumps(data,indent=2))
print('verified source; prepared','final' if FINAL else 'full holdout','fixture',data['source_rows'],{k:sum(v) for k,v in data['counts'].items()},
      'test',data['test_rows'],'sec',round(time.monotonic()-t),flush=True)


Prepared real pilot {'train_0': 926214, 'train_1': 209678, 'train_2': 509018, 'train_3': 411880, 'train_4': 457865, 'train_5': 276084, 'train_6': 155981, 'train_7': 438907, 'train_8': 563122, 'train_9': 671047, 'train_10': 559874, 'train_11': 277765, 'train_12': 344014, 'train_13': 872271, 'train_14': 353613, 'train_15': 392479, 'train_16': 509253, 'train_17': 384412, 'train_18': 629973, 'train_19': 621908, 'train_20': 801121, 'train_21': 384660, 'train_22': 355766, 'train_23': 360226, 'train_24': 338512, 'train_25': 185941, 'train_26': 561149, 'train_27': 372819, 'train_28': 107522, 'train_29': 284527, 'train_30': 1333839, 'train_31': 427651, 'train_32': 648608, 'train_33': 428065, 'train_34': 375085, 'train_35': 882461, 'train_36': 569144, 'train_37': 314555, 'train_38': 436745, 'train_39': 261664, 'train_40': 638454, 'train_41': 398139, 'train_42': 242816, 'train_43': 323428, 'train_44': 451037, 'train_45': 265076, 'train_46': 266508, 'train_47': 506010, 'train_48': 303421, 'train_4

In [3]:
SEED=42
SESSION_HOURS=3.0
WORKERS=2
ARMS=['fd_dml_m4b_gpool_q3_frz_gkd_la_kdpx_zl:50']
FAST=True
DEFER_EVAL=True
SESSION_ROUNDS=2

summary={}
for item in ARMS:
    remaining=SESSION_HOURS-(time.monotonic()-SESSION_T0)/3600
    if remaining<1.0:break
    arm,_,ep=item.partition(':');ep=int(ep or 6)
    folder=OUT/(f'{arm}_s{SEED}'+('' if ep==6 else f'_e{ep}'));folder.mkdir(parents=True,exist_ok=True)
    env=dict(os.environ,PYTHONPATH=f'{PROJ}:{WORK}',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
    env.pop('CUDA_VISIBLE_DEVICES',None)
    command=[sys.executable,str(PROJ/'fd.py'),'--arm',arm,'--seed',str(SEED),'--backend','compiled','--wandb_online',
             '--wandb_project','mobihet-fulldrift','--max_hours',str(round(remaining-.3,3)),'--out',str(OUT),'--stop_after',str(ep),
             '--workers',str(WORKERS),'--clients',str(CLIENTS),'--batch',str(256 if CLIENTS==100 else 512)]+(['--parent',str(PARENT_DIR),'--parent_round',str(PARENT['round'])] if PARENT else [])+(['--fast'] if FAST else [])+(['--final'] if FINAL else [])+(['--defer_eval'] if DEFER_EVAL else [])+(['--session_rounds',str(SESSION_ROUNDS)] if SESSION_ROUNDS else [])
    t=time.monotonic();log=open(folder/'log.txt','a')
    proc=subprocess.Popen(command,cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT)
    print('start',item,SEED,'workers',WORKERS,'remaining_h',round(remaining,2),flush=True)
    while proc.poll() is None:
        if (time.monotonic()-SESSION_T0)/3600>=SESSION_HOURS-.05:proc.terminate()
        time.sleep(5)
    log.close();rp=folder/'result.json'
    result=json.loads(rp.read_text()) if rp.exists() else dict(history=[])
    summary[item]=dict(rc=proc.returncode,epochs=len(result['history']),complete=result.get('complete',False),
                       session_complete=result.get('session_complete',False),evaluation_pending=result.get('evaluation_pending',False),
                       minutes=round((time.monotonic()-t)/60,1))
    (OUT/'fd_summary.json').write_text(json.dumps(summary,indent=2))
    print('finish',item,summary[item],flush=True)
    if proc.returncode:
        print(open(folder/'log.txt').read()[-3000:],flush=True)
        for f in sorted(folder.glob('worker*.log')):print(f.name,f.read_text()[-2000:],flush=True)
(OUT/'fd_summary.json').write_text(json.dumps(summary,indent=2))
print('FullDrift session; gates only after collector audit',json.dumps(summary),flush=True)


start fd_dml_m4b_gpool_q3_frz_gkd_la_kdpx_zl:50 42 workers 2 remaining_h 2.24
finish fd_dml_m4b_gpool_q3_frz_gkd_la_kdpx_zl:50 {'rc': 1, 'epochs': 0, 'complete': False, 'session_complete': False, 'evaluation_pending': False, 'minutes': 2.1}
als for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: minhtriet (21522798-uit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /kaggle/working/runs/fd_dml_m4b_gpool_q3_frz_gkd_la_kdpx_zl_s42_e50/wandb/run-20261020_161557-fd_dml_m4b_gpool_q3_frz_gkd_la_kdpx_zl_s42_e50-50d2d5e4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fd_dml_m4b_gpool_q3_frz_gkd_la_kdpx_zl_s42_e50-fast
wandb: ⭐️ View project at https://wandb.ai/21522798-uit/mobihet-fulldrift
wandb: 🚀 View run at https://wandb.ai/21522798-uit/mobihet-fulldrift/runs/fd_dml_m4b_gpool_q3_fr